# Analyze and save Size Exclusion Chromatography (SEC) spectra

In [ ]:
import sys
from pathlib import Path

%load_ext autoreload
%autoreload 2

REPO_ROOT = Path.cwd().parent

SEC_ANALYSIS = REPO_ROOT / "SEC_analysis"

if SEC_ANALYSIS not in sys.path:
    sys.path.insert(0, str(SEC_ANALYSIS))

In [ ]:
from plot_sec_to_svg import get_series_pairs, render_sec_svg
from sec_raw_to_normalized import run_all_proteins

SEC_DATA_DIR = REPO_ROOT / "data" / "SEC"
OUTPUT_DIR = REPO_ROOT / "outputs" / "SEC"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
protein_data = run_all_proteins(SEC_DATA_DIR, volume_min=0.1, volume_max=25.0)


In [ ]:
list(protein_data.keys())

In [ ]:
# X-axis window per protein (left mL, right mL).
# Normalization is applied within this window so the visible peak reaches 1.0.
# Set to None to use the full data range.
XLIM = {
    "bsa_curves_normalized": (8, 18),
    "Cyt_C_curves_normalized": (15, 22),
    "DNAse_curves_normalized": (8, 23.5),
    "Hemo_curves_normalized": (12, 23),
    "IgG_curves_normalized": (8, 18),
}

for stem, df in protein_data.items():
    pairs = get_series_pairs(df)
    if not pairs:
        print(f"  Skip {stem}: no series found")
        continue
    title = stem.replace("_", " ")
    xlim = XLIM.get(stem)
    f = render_sec_svg(pairs, legend_title=title, xlim=xlim)
    f.savefig(OUTPUT_DIR / f"{stem}.svg", format="svg", bbox_inches="tight")
